In [77]:
import numpy as np
import pandas as pd

import ast

In [78]:
DATE = '2024-08-28'
eval_results_df = pd.read_csv(f"s3://open-jobs-lake/job_quality/outputs/evaluation/JQ_evaluation_results_{DATE}_diff_thresholds.csv")
jq_error_analysis_df = pd.read_csv(f"s3://open-jobs-lake/job_quality/outputs/evaluation/JQ_prediction_errors_{DATE}_diff_thresholds.csv")

eval_results_df['True'] = eval_results_df['True'].apply(lambda x: ast.literal_eval(x))

In [79]:
eval_results_df.head(2)

,Unnamed: 0,JQ_measure_name,False,True,accuracy,macro avg,weighted avg
0,0,L&D,"{'precision': 0.8461538461538461, 'recall': 0....","{'precision': 0.9166666666666666, 'recall': 0....",0.88,"{'precision': 0.8814102564102564, 'recall': 0....","{'precision': 0.8828205128205127, 'recall': 0...."
1,1,CAREER,"{'precision': 0.95, 'recall': 0.97435897435897...","{'precision': 0.9, 'recall': 0.818181818181818...",0.94,"{'precision': 0.925, 'recall': 0.8962703962703...","{'precision': 0.939, 'recall': 0.94, 'f1-score..."


## Show overall metrics per JQ measure

In [80]:
metrics = []
for i, row in eval_results_df.iterrows():
    m = {'jq_measure': row['JQ_measure_name']}
    m.update({k:round(v,3) for k,v in row['True'].items()})
    metrics.append(m)
metrics = pd.DataFrame(metrics)
metrics

,jq_measure,precision,recall,f1-score,support
0,L&D,0.917,0.846,0.880,52.0
1,CAREER,0.900,0.818,0.857,22.0
2,HOURS,0.906,0.983,0.943,59.0
3,FLEX_HOURS,0.756,0.861,0.805,36.0
4,SHIFT,1.000,0.368,0.538,19.0
5,LOC,0.846,0.208,0.333,53.0
6,FLEX_LOC,0.842,0.800,0.821,20.0
7,CONTRACT,0.757,0.700,0.727,40.0
8,LEAVE,0.853,0.967,0.906,30.0
9,COMP,1.000,0.860,0.925,86.0


In [81]:
for i, row in eval_results_df.iterrows():
    print(f"|{row['JQ_measure_name']}|{round(row['True']['support'])}|")

|L&D|52|
|CAREER|22|
|HOURS|59|
|FLEX_HOURS|36|
|SHIFT|19|
|LOC|53|
|FLEX_LOC|20|
|CONTRACT|40|
|LEAVE|30|
|COMP|86|
|PERKS|55|
|CARING|8|
|DISABILITY|2|
|HEALTH|7|
|M_HEALTH|4|
|SPONSORSHIP|2|
|REWARD|3|
|MISC|13|
|AUTONOMY|1|
|SENSE OF PURPOSE|4|
|SOCIAL|22|
|VOICE REPRESENTATION|0|


## Deeper dive
1. Examples of when it does badly for each JQ measure
2. Parent sectors that are better and worse
3. Match threshold and metrics

In [86]:
jq_measure = 'L&D'
error_type = 'FP' # 'FN', 'FP'
for i, row in jq_error_analysis_df[jq_error_analysis_df[jq_measure]==error_type].iterrows():
    print('---')
    print({k:v for k,v in ast.literal_eval(row['jq_sentences']).items()})
    
    # print({k:v for k,v in ast.literal_eval(row['jq_sentences']).items() if jq_measure in v})
    print('..')
    # print([v for v in ast.literal_eval(row['pred_ngram_matched']) if v[3] == jq_measure] )
    print([v for v in ast.literal_eval(row['pred_ngram_matched'])] )

---
{'There are currently numerous opportunities for work in different schools in the local area and would possibly suit a teacher that  is looking for flexibility in their work and life balance.': ['FLEX_HOURS'], 'There are currently numerous opportunities for work in different schools in the local area and would possibly suit a teacher that is  looking for flexibility in their work and life balance.': ['FLEX_HOURS'], 'You can also utilise the numerous benefits we offer; - Your very own dedicated consultant - A variety of daily, short and long term positions to suit your needs - Competitive rates of pay - No need for completion of time sheets - Email   SMS confirmation for all bookings - £75 reward scheme for each Teacher you introduce - Control if your own teaching diary; flexibility': ['FLEX_HOURS', 'CONTRACT', 'COMP', 'PERKS']}
..
[(' flexibility', 0.7224345803260803, 'flexible working', 'FLEX_HOURS'), (' short and long term positions', 0.6011300086975098, 'long term', 'CONTRACT'),

In [83]:
jq_error_analysis_df.groupby('parent_sector')['n_incorrect'].mean()

parent_sector
Accountancy                    2.333333
Accountancy (Qualified)        2.833333
Admin, Secretarial &amp; PA    1.428571
Banking                        0.000000
Charity &amp; Voluntary        2.000000
Construction &amp; Property    2.500000
Education                      2.666667
Energy                         2.000000
Engineering                    4.500000
Estate Agency                  2.000000
FMCG                           0.000000
Health &amp; Medicine          2.250000
Hospitality &amp; Catering     2.444444
Human Resources                1.750000
IT &amp; Telecoms              1.666667
Marketing &amp; PR             2.666667
Motoring &amp; Automotive      3.500000
Recruitment Consultancy        3.000000
Retail                         1.937500
Sales                          3.500000
Social Care                    3.300000
Transport &amp; Logistics      1.857143
Name: n_incorrect, dtype: float64

## Threshold

In [87]:
DATE = '2024-08-28'
jq_error_analysis_df_no_thresh = pd.read_csv(f"s3://open-jobs-lake/job_quality/outputs/evaluation/JQ_prediction_errors_{DATE}_no_thresh.csv")

In [88]:
from sklearn.metrics import (
    classification_report,
)
import altair as alt

In [89]:
jq_cols = [
    "L&D",
    "CAREER",
    "HOURS",
    "FLEX_HOURS",
    "SHIFT",
    "LOC",
    "FLEX_LOC",
    "CONTRACT",
    "LEAVE",
    "COMP",
    "PERKS",
    "CARING",
    "DISABILITY",
    "HEALTH",
    "M_HEALTH",
    "SPONSORSHIP",
    "REWARD",
    "MISC",
    "AUTONOMY",
    "SENSE OF PURPOSE",
    "SOCIAL",
    "VOICE REPRESENTATION",
]

In [90]:
def get_thresh_result(x, cs_threshold, jq_measure):
    if pd.notnull(x):
        x = ast.literal_eval(x)
        res = [cat for _, thresh, _, cat in x if ((thresh>=cs_threshold) & (cat==jq_measure))]
        if len(res) ==0:
            return False
        else:
            return True
    else:
        return False
        


In [91]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

In [92]:
class_rep_per_jq = []
for jq_measure in jq_cols:
    truth_list = jq_error_analysis_df_no_thresh[jq_measure].isin(["TP", "FN"]).tolist()
    for cs_threshold in [0.3,0.35,0.4,0.45,0.5,0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]:
        pred_list = jq_error_analysis_df_no_thresh['pred_ngram_matched'].apply(lambda x: get_thresh_result(x, cs_threshold, jq_measure))
        class_rep = classification_report(
            truth_list, pred_list, output_dict=True
        )
        if 'True' not in class_rep:
            class_rep['True'] = {'recall': None, 'precision': None, 'f1-score': None, 'support': None}
        class_rep_per_jq.append(
            {'jq_measure': jq_measure,
             'threshold': cs_threshold,
             'recall': class_rep['True']['recall'],
             'precision': class_rep['True']['precision'],
             'f1-score': class_rep['True']['f1-score'],
             'support': class_rep['True']['support'],
            })

class_rep_per_jq_df = pd.DataFrame(class_rep_per_jq)

In [93]:
jq_measures = class_rep_per_jq_df[((class_rep_per_jq_df['threshold']==0.55) & (class_rep_per_jq_df['support']>10))]['jq_measure'].tolist()
jq_measures = [j for j in jq_measures if j not in ['MISC', 'SOCIAL']]
filtered_jq_data = class_rep_per_jq_df[class_rep_per_jq_df['jq_measure'].isin(jq_measures[0:5])]
filtered_jq_data2 = class_rep_per_jq_df[class_rep_per_jq_df['jq_measure'].isin(jq_measures[5:])]

rec_plot = alt.Chart(filtered_jq_data, title="Recall (dashed), precision (bold)").mark_line(strokeDash=[1,1], strokeWidth=5).encode(
    x='threshold', y='recall', color='jq_measure', 
)
prec_plot = alt.Chart(filtered_jq_data).mark_line(strokeWidth=5).encode(
    x='threshold', y='precision', color='jq_measure'
)

rec_plot2 = alt.Chart(filtered_jq_data2, title="Recall (dashed), precision (bold)").mark_line(strokeDash=[1,1], strokeWidth=5).encode(
    x='threshold', y='recall', color='jq_measure', 
)
prec_plot2 = alt.Chart(filtered_jq_data2).mark_line(strokeWidth=5).encode(
    x='threshold', y='precision', color='jq_measure'
)

((rec_plot+prec_plot) | (rec_plot2+prec_plot2)).resolve_scale(color='independent')

alt.HConcatChart(...)

Changes to threshold that would make results more precise and not effect recall:
- CAREER: 0.6
- FLEX_HOURS: 0.65
- HOURS: 0.6
- FLEX_LOC: 0.6
- LEAVE: 0.65
- CONTRACT: 0.5
- LOC: 0.6
- OTHER: 0.55

In [94]:
def find_best_compromise_threshold(df):
    results = []
    measures = df['jq_measure'].unique()
    
    for measure in measures:
        subset = df[df['jq_measure'] == measure].dropna(subset=['recall', 'precision'])
        # You need to make sure these aren't 0 because for some measures there is a crossover point at 0.
        # We don't want to accidentally optimise for 0 recall and 0 precision...
        subset = subset[(subset['recall'] >0) & (subset['precision'] > 0)]
        
        if subset.empty:
            results.append({
                "jq_measure": measure,
                "best_compromise_threshold": None
            })
            continue
        
        # Find the intersection point between recall and precision
        diffs = np.abs(subset['recall'] - subset['precision'])
        # Find the point with the smallest difference between these two values
        best_idx = diffs.idxmin()
        
        best_threshold = subset.loc[best_idx, 'threshold']
        
        results.append({
            "jq_measure": measure,
            "best_compromise_threshold": best_threshold
        })
    
    return pd.DataFrame(results)

compromise_thresholds = find_best_compromise_threshold(class_rep_per_jq_df)
compromise_thresholds

,jq_measure,best_compromise_threshold
0,L&D,0.50
1,CAREER,0.55
2,HOURS,0.65
3,FLEX_HOURS,0.70
4,SHIFT,0.30
5,LOC,0.45
6,FLEX_LOC,0.60
7,CONTRACT,0.45
8,LEAVE,0.70
9,COMP,0.40


In [95]:
# Check that the threshold we've chosen for 'OTHER' is reasonable
median_thresh = compromise_thresholds['best_compromise_threshold'].median()
print(f'The most sensible threshold for the "OTHER" category is {median_thresh}')

The most sensible threshold for the "OTHER" category is 0.55
